In [1]:
!pip -q install -U \
    "transformers>=4.45,<5" \
    "trl==0.12.2" \
    "peft==0.13.2" \
    "datasets==3.1.0" \
    accelerate \
    sentence-transformers \
    pypdf \
    scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 15.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 320.7/320.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 79.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 402.7/402.7 kB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 80.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.7/146.7 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [3]:
import os
import json
import textwrap
import numpy as np
import torch

from pathlib import Path
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments
)

from datasets import Dataset
from peft import LoraConfig
from trl import SFTTrainer

print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Warning: Training on CPU will be slow.")

PyTorch: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


In [4]:
MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=(
        torch.float16
        if torch.cuda.is_available()
        else torch.float32
    )
)

if torch.cuda.is_available():
    model = model.to("cuda")

model.config.use_cache = True
model.eval()

print("Loaded:", MODEL_NAME)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Loaded: Qwen/Qwen2.5-1.5B-Instruct


In [5]:
assistant_preferences = {
    "purpose": "General-purpose AI assistant",
    "expertise_level": "Intermediate",
    "response_style": "Clear, practical and structured",
    "response_length": "Moderate detail",
    "preferred_format": "Explanation followed by numbered steps",
    "language": "English",
    "special_instructions": (
        "Give examples when useful. "
        "Admit uncertainty instead of inventing facts."
    )
}

def preferences_to_text(preferences):
    return "\n".join(
        f"{key.replace('_', ' ').title()}: {value}"
        for key, value in preferences.items()
    )

SYSTEM_PROMPT = """
You are PersonalAI, a helpful personalized AI assistant.

Follow the user's response preferences.
Use uploaded documents as evidence when relevant.
Do not invent facts about the user or documents.
Clearly distinguish retrieved facts from general knowledge.
If the available information is insufficient, say so.
"""

print(preferences_to_text(assistant_preferences))

Purpose: General-purpose AI assistant
Expertise Level: Intermediate
Response Style: Clear, practical and structured
Response Length: Moderate detail
Preferred Format: Explanation followed by numbered steps
Language: English
Special Instructions: Give examples when useful. Admit uncertainty instead of inventing facts.


In [6]:
def generate_response(question, context="", max_new_tokens=220):
    preference_text = preferences_to_text(
        assistant_preferences
    )

    user_message = f"""
USER PREFERENCES:
{preference_text}

RELEVANT DOCUMENT CONTEXT:
{context if context else "No document context supplied."}

QUESTION:
{question}

Answer according to the preferences. Use document context when relevant.
"""

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_message}
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(next(model.parameters()).device)

    model.eval()

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_tokens = output[
        0, inputs["input_ids"].shape[1]:
    ]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    ).strip()


# First baseline test
question = "Explain what RAG is and give me an example."

print(generate_response(question))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


RAG stands for Retrieval-Augmented Generation. It is a type of machine learning model that combines retrieval-based text search with generative models like transformers to improve language generation tasks. The core idea behind RAG is to leverage pre-existing knowledge in the form of large text corpora to enhance the quality and relevance of generated text.

### Example:

Imagine you're writing a blog post on "The Benefits of Meditation." You want your content to be well-researched and engaging. To achieve this, you could use RAG to generate the initial draft of your post based on a set of predefined topics related to meditation (e.g., mindfulness techniques, stress relief, etc.). Then, you would use a retrieval system to find additional articles, studies, and expert opinions that support these points but go beyond the basic framework provided by the generative model.

For instance, if you were discussing the benefits of meditation, the generative model might suggest something along th

In [7]:
from google.colab import files

uploaded = files.upload()

documents = []

for filename in uploaded:
    extension = Path(filename).suffix.lower()

    if extension in [".txt", ".md"]:
        text = Path(filename).read_text(
            encoding="utf-8",
            errors="ignore"
        )

    elif extension == ".pdf":
        reader = PdfReader(filename)
        text = "\n".join(
            page.extract_text() or ""
            for page in reader.pages
        )

    else:
        print("Skipping unsupported file:", filename)
        continue

    if text.strip():
        documents.append({
            "source": filename,
            "text": text
        })

print("Documents loaded:", len(documents))
for doc in documents:
    print(doc["source"], "characters:", len(doc["text"]))

Saving summary-notes (1).pdf to summary-notes (1).pdf
Documents loaded: 1
summary-notes (1).pdf characters: 17382


In [8]:
def split_into_chunks(text, chunk_size=180, overlap=35):
    words = text.split()
    chunks = []

    start = 0
    while start < len(words):
        end = min(start + chunk_size, len(words))
        chunk = " ".join(words[start:end])

        if chunk.strip():
            chunks.append(chunk)

        if end == len(words):
            break

        start = end - overlap

    return chunks


chunks = []

for doc in documents:
    for i, chunk in enumerate(split_into_chunks(doc["text"])):
        chunks.append({
            "source": doc["source"],
            "chunk_id": i,
            "text": chunk
        })

print("Total document chunks:", len(chunks))

Total document chunks: 17


In [9]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

if chunks:
    chunk_embeddings = embedding_model.encode(
        [chunk["text"] for chunk in chunks],
        normalize_embeddings=True,
        show_progress_bar=True
    )
else:
    chunk_embeddings = np.empty((0, 384))

print("Embedding matrix shape:", chunk_embeddings.shape)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding matrix shape: (17, 384)


In [10]:
def retrieve_documents(question, top_k=3):
    if not chunks:
        return []

    question_embedding = embedding_model.encode(
        [question],
        normalize_embeddings=True
    )

    scores = cosine_similarity(
        question_embedding,
        chunk_embeddings
    )[0]

    best_indices = np.argsort(scores)[::-1][:top_k]

    results = []

    for idx in best_indices:
        if scores[idx] > 0.15:
            results.append({
                **chunks[idx],
                "score": float(scores[idx])
            })

    return results


def build_rag_context(question):
    results = retrieve_documents(question)

    context = "\n\n".join(
        f"[Source: {item['source']}, "
        f"passage {item['chunk_id']}]\n{item['text']}"
        for item in results
    )

    return context, results


# Test retrieval
test_question = "What are the main ideas in my documents?"
context, results = build_rag_context(test_question)

print(context if context else "No relevant passages found.")

[Source: summary-notes (1).pdf, passage 1]
impacting the ethical and practical dimensions of research. By examining concepts like the "metamind," skepticism, axiology, and feminist epistemologies, we gain a comprehensive understanding of the complex landscape of knowledge and its implications for scholarly inquiry. Core Concepts Term Definition Significance Epistemology The branch of philosophy concerned with the theory of knowledge, including its nature, scope, and justification. It is fundamental to research as it dictates how knowledge is acquired, validated, and understood, influencing research design and methodology. Metamind The collective sum of all knowledge ever created and to be created by the Homo sapiens species. Represents the vast, evolving ocean of human knowledge, highlighting that individual minds contribute droplets to Term Definition Significance this shared intellectual heritage. Paradigm A conceptual framework or set of assumptions, theories, and practices that con

In [11]:
def ask_with_rag(question):
    context, retrieved = build_rag_context(question)

    if not context:
        answer = generate_response(question)
    else:
        answer = generate_response(question, context=context)

    sources = list(dict.fromkeys(
        item["source"] for item in retrieved
    ))

    return answer, sources


question = "Summarize the most relevant information in my documents."

rag_answer, rag_sources = ask_with_rag(question)

print("ANSWER:\n", rag_answer)
print("\nRETRIEVED SOURCE FILES:", rag_sources)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(


ANSWER:
 To summarize the most relevant information from your documents:

1. **Epistemology**: 
   - Epistemology is the branch of philosophy concerned with the theory of knowledge, including its nature, scope, and justification.
   - It is fundamental to research as it dictates how knowledge is acquired, validated, and understood, influencing research design and methodology.
   - Different epistemological stances lead to varied approaches to knowledge creation and validation, impacting the ethical and practical dimensions of research.

2. **Metamind**:
   - The metamind refers to the collective sum of all knowledge ever created and to be created by the Homo sapiens species.
   - It highlights that individual minds contribute "droplets" to this shared intellectual heritage, representing the vast, evolving ocean of human knowledge.

3. **Paradigm**:
   - A paradigm is a conceptual framework or set of assumptions, theories, and practices that constitute a way of viewing reality.
   - In 

In [12]:
evaluation_questions = [
    "Explain RAG in simple terms.",
    "What information in my documents is relevant to my question?",
    "Give me a structured answer with practical steps."
]

baseline_results = []

for q in evaluation_questions:
    base_answer = generate_response(q)
    rag_answer, sources = ask_with_rag(q)

    baseline_results.append({
        "question": q,
        "base_answer": base_answer,
        "rag_answer": rag_answer,
        "rag_sources": sources
    })

print("Saved baseline and RAG results:", len(baseline_results))

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarni

Saved baseline and RAG results: 3


In [13]:
training_examples = [
    {
        "instruction": "Explain a technical concept.",
        "response": (
            "Start with a simple definition. Explain how it works, "
            "give a practical example, and finish with the key takeaway."
        )
    },
    {
        "instruction": "Answer a question using uploaded documents.",
        "response": (
            "Identify the relevant document evidence first. "
            "Explain the answer clearly and identify the source. "
            "If the evidence is insufficient, say what is missing."
        )
    },
    {
        "instruction": "Help someone learn a new topic.",
        "response": (
            "Break the topic into manageable steps. Explain unfamiliar "
            "terms, include a small example, and suggest a practice task."
        )
    },
    {
        "instruction": "Compare two technical approaches.",
        "response": (
            "Explain each approach, compare its benefits and limitations, "
            "and recommend one only after considering the requirements."
        )
    },
    {
        "instruction": "Answer a question when information is missing.",
        "response": (
            "State which information is unavailable. Avoid inventing facts "
            "and ask a focused follow-up question if necessary."
        )
    },
    {
        "instruction": "Give a concise answer.",
        "response": (
            "Answer the main question directly, provide the essential "
            "reason, and include one useful example when appropriate."
        )
    },
    {
        "instruction": "Create a step-by-step plan.",
        "response": (
            "State the goal, divide the work into ordered steps, "
            "identify any prerequisites, and describe how to check progress."
        )
    },
    {
        "instruction": "Explain a difficult idea to a beginner.",
        "response": (
            "Use plain language, define technical terms, explain one idea "
            "at a time, and use a familiar analogy when helpful."
        )
    },
    {
        "instruction": "Summarize a long document.",
        "response": (
            "Identify the central idea, list the main supporting points, "
            "preserve important qualifications, and avoid adding claims "
            "that are not in the source."
        )
    },
    {
        "instruction": "Provide a recommendation.",
        "response": (
            "Clarify the user's objective, compare relevant options, "
            "explain the trade-offs, and state the assumptions behind "
            "the recommendation."
        )
    },
    {
        "instruction": "Respond when uncertain.",
        "response": (
            "Clearly distinguish known information from uncertainty. "
            "Do not present guesses as verified facts."
        )
    },
    {
        "instruction": "Format an answer for easy reading.",
        "response": (
            "Use a direct opening, descriptive headings where useful, "
            "numbered steps for procedures, and a brief conclusion."
        )
    }
]

def format_training_example(example):
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT
        },
        {
            "role": "user",
            "content": example["instruction"]
        },
        {
            "role": "assistant",
            "content": example["response"]
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False
    )


train_dataset = Dataset.from_dict({
    "text": [
        format_training_example(example)
        for example in training_examples
    ]
})

print("Training examples:", len(train_dataset))
print(train_dataset[0]["text"])

Training examples: 12
<|im_start|>system

You are PersonalAI, a helpful personalized AI assistant.

Follow the user's response preferences.
Use uploaded documents as evidence when relevant.
Do not invent facts about the user or documents.
Clearly distinguish retrieved facts from general knowledge.
If the available information is insufficient, say so.
<|im_end|>
<|im_start|>user
Explain a technical concept.<|im_end|>
<|im_start|>assistant
Start with a simple definition. Explain how it works, give a practical example, and finish with the key takeaway.<|im_end|>



In [14]:
from peft import TaskType

# Clear the inference cache before training.
if torch.cuda.is_available():
    torch.cuda.empty_cache()

model.config.use_cache = False

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj"
    ],
    bias="none"
)

training_args = TrainingArguments(
    output_dir="./personalai_lora_results",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    save_strategy="no",
    report_to="none",
    fp16=torch.cuda.is_available()
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
    peft_config=lora_config,
    dataset_text_field="text",
    max_seq_length=512
)

trainer.model.print_trainable_parameters()

training_output = trainer.train()

print("Fine-tuning completed!")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_deprecation.py:100: FutureWarning: Deprecated argument(s) used in '__init__': dataset_text_field, max_seq_length. Will not be supported from version '0.13.0'.

Deprecated positional argument(s) used in SFTTrainer, please use the SFTConfig to set these arguments instead.
  warnings.warn(message, FutureWarning)
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:300: UserWarning: You passed a `max_seq_length` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/trl/trainer/sft_trainer.py:328: UserWarning: You passed a `dataset_text_field` argument to the SFTTrainer, the value you passed will override the one in the `SFTConfig`.
  warnings.warn(


Map:   0%|          | 0/12 [00:00<?, ? examples/s]

trainable params: 9,232,384 || all params: 1,552,946,688 || trainable%: 0.5945


Step,Training Loss
1,3.871200
2,3.580400
3,3.223300
4,2.773000
5,2.440700
6,2.235300
7,2.112300
8,1.802500
9,1.785800


Fine-tuning completed!


In [15]:
ADAPTER_PATH = "./personalai_lora_adapter"

trainer.model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)

print("Adapter saved to:", ADAPTER_PATH)

Adapter saved to: ./personalai_lora_adapter


In [16]:
# Switch inference to the fine-tuned model.
model = trainer.model
model.config.use_cache = True
model.eval()

print("Fine-tuned model is ready.")

Fine-tuned model is ready.


In [17]:
comparison_results = []

for item in baseline_results:
    q = item["question"]

    fine_tuned_answer = generate_response(q)

    fine_tuned_rag_answer, sources = ask_with_rag(q)

    comparison_results.append({
        "question": q,
        "base_llm": item["base_answer"],
        "rag": item["rag_answer"],
        "fine_tuned": fine_tuned_answer,
        "fine_tuned_rag": fine_tuned_rag_answer,
        "sources": ", ".join(sources)
    })

import pandas as pd

comparison_df = pd.DataFrame(comparison_results)

pd.set_option("display.max_colwidth", 180)

display(comparison_df)

/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.7` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:595: UserWarning: `do_sample` is set to `False`. However, `top_p` is set to `0.8` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_p`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:612: UserWarning: `do_sample` is set to `False`. However, `top_k` is set to `20` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `top_k`.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/transformers/generation/configuration_utils.py:590: UserWarni

,question,base_llm,rag,fine_tuned,fine_tuned_rag,sources
0,Explain RAG in simple terms.,RAG stands for Retrieval-Augmented Generation. It's an advanced machine learning technique that combines two powerful tools: retrieval and generation.\n\n1. **Retrieval**: This...,RAG stands for Retrieval-Augmented Generation. It's an advanced machine learning technique that combines two powerful tools: retrieval and generation.\n\n1. **Retrieval**: This...,RAG stands for Retrieval-Augmented Generation. It combines two powerful techniques: retrieval-based question answering (RQA) and generative language models like GPT-3. Here’s h...,RAG stands for Retrieval-Augmented Generation. It combines two powerful techniques: retrieval-based question answering (RQA) and generative language models like GPT-3. Here’s h...,
1,What information in my documents is relevant to my question?,"To determine which information in your documents is relevant to your question, I will follow these steps:\n\n1. Review each document for any headings, subheadings, or keywords ...","The information in your documents that is most relevant to your question is:\n\n- **Introduction to Epistemology**: It discusses the importance of epistemology in research, exp...","To determine which information in your documents is relevant to your question, I will follow these steps:\n\n1. Review each document for any headings, subheadings, or keywords ...","The information in your documents that is relevant to your question includes:\n\n- The foundational role of epistemology in research, emphasizing its importance in understandin...",summary-notes (1).pdf
2,Give me a structured answer with practical steps.,"To provide you with a structured answer that includes practical steps, I'll need more specific details about what kind of assistance or task you're looking for. Could you pleas...","To provide you with a structured answer that includes practical steps, I'll need more specific details about what kind of assistance or task you're looking for. Could you pleas...","To provide you with a structured answer that includes practical steps, I'll need more specific details about what kind of assistance you're looking for. Could you please specif...","To provide you with a structured answer that includes practical steps, I'll need more specific details about what kind of assistance you're looking for. Could you please specif...",


In [18]:
criteria = [
    "Correctness",
    "Relevance",
    "Groundedness",
    "Personalization",
    "Response clarity"
]

evaluation = pd.DataFrame({
    "Criterion": criteria,
    "Base LLM (0-5)": [None] * len(criteria),
    "RAG (0-5)": [None] * len(criteria),
    "Fine-tuned (0-5)": [None] * len(criteria),
    "Fine-tuned + RAG (0-5)": [None] * len(criteria)
})

display(evaluation)

,Criterion,Base LLM (0-5),RAG (0-5),Fine-tuned (0-5),Fine-tuned + RAG (0-5)
0,Correctness,None,None,None,None
1,Relevance,None,None,None,None
2,Groundedness,None,None,None,None
3,Personalization,None,None,None,None
4,Response clarity,None,None,None,None


In [19]:
comparison_df.to_csv(
    "personalai_comparison.csv",
    index=False
)

evaluation.to_csv(
    "personalai_evaluation.csv",
    index=False
)

print("Evaluation files saved.")

Evaluation files saved.


In [20]:
from google.colab import files

files.download("personalai_comparison.csv")
files.download("personalai_evaluation.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [25]:
import os

print("Comparison CSV exists:",
      os.path.exists("personalai_comparison.csv"))

print("Evaluation CSV exists:",
      os.path.exists("personalai_evaluation.csv"))

print("Current directory:", os.getcwd())

Comparison CSV exists: True
Evaluation CSV exists: True
Current directory: /content


In [26]:
from google.colab import files
import os
import zipfile

with zipfile.ZipFile("personalai_results.zip", "w") as z:
    for filename in [
        "personalai_comparison.csv",
        "personalai_evaluation.csv"
    ]:
        if os.path.exists(filename):
            z.write(filename)

files.download("personalai_results.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [27]:
import os
import zipfile

ADAPTER_PATH = "./personalai_lora_adapter"

print("Adapter folder exists:", os.path.exists(ADAPTER_PATH))

if os.path.exists(ADAPTER_PATH):
    print("Files:", os.listdir(ADAPTER_PATH))

Adapter folder exists: True
Files: ['adapter_model.safetensors', 'added_tokens.json', 'special_tokens_map.json', 'tokenizer_config.json', 'merges.txt', 'README.md', 'vocab.json', 'tokenizer.json', 'adapter_config.json']


In [30]:
from google.colab import files

with zipfile.ZipFile("personalai_lora_adapter.zip", "w",
                     zipfile.ZIP_DEFLATED) as z:
    for root, dirs, filenames in os.walk(ADAPTER_PATH):
        for filename in filenames:
            path = os.path.join(root, filename)
            z.write(path, arcname=path)

files.download("personalai_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [31]:
import os

ADAPTER_PATH = "/content/personalai_lora_adapter"

print("Adapter folder exists:", os.path.isdir(ADAPTER_PATH))

if os.path.isdir(ADAPTER_PATH):
    print("Files inside adapter folder:")
    print(os.listdir(ADAPTER_PATH))

Adapter folder exists: True
Files inside adapter folder:
['adapter_model.safetensors', 'added_tokens.json', 'special_tokens_map.json', 'tokenizer_config.json', 'merges.txt', 'README.md', 'vocab.json', 'tokenizer.json', 'adapter_config.json']


In [32]:
import os
import zipfile

ADAPTER_PATH = "/content/personalai_lora_adapter"
ZIP_PATH = "/content/personalai_lora_adapter.zip"

if not os.path.isdir(ADAPTER_PATH):
    print("Adapter folder not found. We need to check the training step.")
else:
    with zipfile.ZipFile(
        ZIP_PATH, "w", zipfile.ZIP_DEFLATED
    ) as z:
        for root, dirs, filenames in os.walk(ADAPTER_PATH):
            for filename in filenames:
                full_path = os.path.join(root, filename)
                relative_path = os.path.relpath(
                    full_path, "/content"
                )
                z.write(full_path, relative_path)

    print("ZIP exists:", os.path.exists(ZIP_PATH))
    print("ZIP size:", os.path.getsize(ZIP_PATH), "bytes")
    print("ZIP contents:")

    with zipfile.ZipFile(ZIP_PATH, "r") as z:
        print(z.namelist())

ZIP exists: True
ZIP size: 37919586 bytes
ZIP contents:
['personalai_lora_adapter/adapter_model.safetensors', 'personalai_lora_adapter/added_tokens.json', 'personalai_lora_adapter/special_tokens_map.json', 'personalai_lora_adapter/tokenizer_config.json', 'personalai_lora_adapter/merges.txt', 'personalai_lora_adapter/README.md', 'personalai_lora_adapter/vocab.json', 'personalai_lora_adapter/tokenizer.json', 'personalai_lora_adapter/adapter_config.json']


In [34]:
from google.colab import files

files.download("/content/personalai_lora_adapter.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [35]:

!pip -q install flask


In [36]:

import secrets
import threading
import hmac
import torch
from flask import Flask, request, jsonify

# Keep this token private
API_TOKEN = secrets.token_urlsafe(32)

api = Flask("PersonalAI_Inference_API")

@api.route("/health", methods=["GET"])
def health():
    supplied = request.headers.get("Authorization", "")
    if not hmac.compare_digest(supplied, f"Bearer {API_TOKEN}"):
        return jsonify({"error": "Unauthorized"}), 401

    return jsonify({
        "status": "ready",
        "model": "Qwen/Qwen2.5-1.5B-Instruct",
        "gpu": torch.cuda.is_available()
    })

@api.route("/generate", methods=["POST"])
def generate():
    supplied = request.headers.get("Authorization", "")
    if not hmac.compare_digest(supplied, f"Bearer {API_TOKEN}"):
        return jsonify({"error": "Unauthorized"}), 401

    data = request.get_json(silent=True) or {}
    question = data.get("question", "")
    context = data.get("context", "")

    if not isinstance(question, str) or not question.strip():
        return jsonify({"error": "Question is required"}), 400
    if not isinstance(context, str):
        return jsonify({"error": "Context must be text"}), 400
    if len(question) > 4000 or len(context) > 12000:
        return jsonify({"error": "Input is too long"}), 413

    messages = [
        {
            "role": "system",
            "content": (
                "You are PersonalAI, a helpful assistant. "
                "Use document context when relevant. "
                "Do not invent facts about documents. "
                "If context is insufficient, say so."
            )
        },
        {
            "role": "user",
            "content": (
                f"DOCUMENT CONTEXT:\n{context or 'No documents supplied.'}"
                f"\n\nQUESTION:\n{question}"
            )
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=4096
    ).to(next(model.parameters()).device)

    model.eval()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    new_tokens = output[0, inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(
        new_tokens, skip_special_tokens=True
    ).strip()

    return jsonify({"answer": answer})

def run_api():
    api.run(
        host="127.0.0.1",
        port=7860,
        debug=False,
        use_reloader=False
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

print("PersonalAI API started on port 7860.")
print("Your private API token (keep it secret):")
print(API_TOKEN)


PersonalAI API started on port 7860.
Your private API token (keep it secret):
4sCuMIqogYSp8g4RBHqrGv5eFSAdE2tr1rQc5C9NGfA


In [37]:

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /content/cloudflared
!chmod +x /content/cloudflared

import subprocess
import time

cloudflared_log = open("/content/cloudflared.log", "w")

cloudflared_proc = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--url", "http://127.0.0.1:7860"],
    stdout=cloudflared_log,
    stderr=subprocess.STDOUT
)

time.sleep(8)

print(open("/content/cloudflared.log").read())


2026-10-13T09:35:17Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-13T09:35:17Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-13T09:35:22Z INF +--------------------------------------------------------------------------------------------+
2026-10-13T09:35:22Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-13T09:35:22Z INF |  https://her-periodically-etc-nations.trycloudflare.co